# Quark: build a data mix on Google Colab (CPU)

Streams each source of a mix recipe, keeps the documents its filter accepts, then trains one tokenizer and tokenizes every source (`docs/domain_data_survey.md`, plan §18). Everything is saved to **Google Drive**, where the training and compare notebooks pick it up.

**No GPU needed:** *Runtime → Change runtime type → CPU*, so this doesn't use GPU quota. Then *Runtime → Run all*.

**Long scans are resumable.** Collection saves its stream position every ~20,000 documents. If the session ends, *Run all* again and it continues where it stopped. Keep the browser tab open while it runs.

## 2. Google Drive (persistent storage)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
DRIVE_ROOT = Path("/content/drive/MyDrive/quark")  # models, checkpoints and cached data live here
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
print("Drive folder:", DRIVE_ROOT)

## 3. Install Quark

In [ ]:
QUARK_REF = "main"  # a branch, tag or commit of the quark repo
%pip install -q "git+https://github.com/cpalmquist19/quark.git@{QUARK_REF}"

## Settings
The mix recipe: sources, filters and token targets. Training weights are chosen later, in the training config or compare spec. Changing a source's filter after collecting it means deleting its folder under `raw/` first.

In [ ]:
DATA_ROOT = DRIVE_ROOT / "data"

# A raw string, because the filters' regexes contain backslashes.
RECIPE_YAML = r'''
# Data mix v1 for the chatbot target (plan §18, docs/domain_data_survey.md). Sized for the ablation ladder
# (ablation-s / ablation-m); the target-size run will need a bigger collection.
#
#   quark data collect mixes/chatbot-v1.yaml       CPU only, resumable; ~3-5 h (one pass over FineWeb-Edu
#                                                   feeds all four FineWeb sources; neurodivergence is the
#                                                   rarest, so it decides how long the scan runs)
#   quark data prepare-mix mixes/chatbot-v1.yaml   shared tokenizer + per-source token files
# On Colab: notebooks/quark_data_colab.ipynb (a CPU runtime is enough).
#
# Training picks the weights (data.weights in the config or compare spec), not this file. A starting point:
#   general 0.70, education 0.10, neurodivergence 0.08, early_childhood 0.07, children_stories 0.05
# These are guesses to be tested: the domain weights are themselves a compare experiment.

name: chatbot-v1
tokenizer: {type: train, size: 8192}
val_fraction: 0.005

sources:
  general:
    note: General educational English, the base of the mix.
    source: HuggingFaceFW/fineweb-edu
    subset: sample-10BT
    tokens: 300000000

  education:
    note: Teaching and learning (teacher-facing material); ~4.5% of FineWeb-Edu.
    source: HuggingFaceFW/fineweb-edu
    subset: sample-10BT
    tokens: 60000000
    filter: {domain: education}

  early_childhood:
    note: >-
      Knowledge about early childhood: development, teaching young children, parenting (option b, decided
      2026-10-10). ~0.86% of FineWeb-Edu, mostly adult-facing.
    source: HuggingFaceFW/fineweb-edu
    subset: sample-10BT
    tokens: 25000000
    filter: {domain: early_childhood}

  neurodivergence:
    note: >-
      Neurodivergence, curated toward a neurodiversity-affirming perspective (decided 2026-10-10; children will
      talk to the model). ~0.74% of FineWeb-Edu, mostly medical/parent-facing, so deficit- and cure-framed text
      is left out. The lists below are a first pass for review, not a final word: excluding a phrase also drops
      articles that quote it to argue against it.
    source: HuggingFaceFW/fineweb-edu
    subset: sample-10BT
    tokens: 25000000
    filter:
      domain: neurodivergence
      exclude_sites:
        - autismspeaks.org     # criticized by autistic self-advocacy groups for cure/deficit framing
        - ageofautism.com      # vaccine-autism misinformation
        - generationrescue.org # vaccine-autism misinformation, "recovery" framing
      exclude_patterns:
        - '\bsuffer(?:s|ing|ed)? (?:from|with) (?:autism|adhd|dyslexia|asperger)'
        - '\b(?:afflicted|stricken) (?:with|by) (?:autism|adhd)'
        - '\bcur(?:e|es|ing|ed) (?:for |of )?(?:autism|adhd)'
        - '\brecover(?:y|ed|ing)? from autism'
        - '\bdefeat autism'
        - '\bautism epidemic'
        - '\bvaccines? (?:cause|causes|caused) autism'
        - '\bchelation'

  children_stories:
    note: >-
      Child-voiced synthetic stories (Cosmopedia, format story_children): simple, warm language for a model
      children talk to. Formulaic, so keep its weight low.
    source: HuggingFaceTB/cosmopedia
    subset: stories
    tokens: 30000000
    filter:
      columns: {format: [story_children]}
'''

In [ ]:
import yaml
from quark.data.mix import MixRecipe, collect_status, mix_dir

recipe = MixRecipe.model_validate(yaml.safe_load(RECIPE_YAML))
MIX_DIR = mix_dir(recipe, DATA_ROOT)
print(f"mix {recipe.name} -> {MIX_DIR}")
for name, st in collect_status(recipe, DATA_ROOT).items():
    print(f"  {name:18s} {st['tokens'] / 1e6:8,.1f}M / {st['target'] / 1e6:,.0f}M tokens  {'done' if st['done'] else ''}")

## Collect
One pass per stream: all FineWeb-Edu sources are filled from the same scan. Progress prints at every save.

In [ ]:
from quark.data.mix import collect

status = collect(recipe, DATA_ROOT)

## Prepare
Trains the shared tokenizer, then tokenizes each source into its own train/val files.

In [ ]:
from quark.data.mix import prepare_mix

if (MIX_DIR / "meta.json").exists():
    print("already prepared:", MIX_DIR)
else:
    meta = prepare_mix(recipe, DATA_ROOT, progress=lambda name, d, t: d % 50_000 == 0 and print(f"  {name}: {d:,} docs"))
    for name, src in meta["sources"].items():
        print(f"{name:18s} train {src['train_tokens'] / 1e6:7,.1f}M tokens ({src['train_docs']:,} docs) | val {src['val_tokens'] / 1e6:.2f}M")

Done. To train on this mix, set `data.mix` to the mix name (and the same `data.tokenizer`) in a model config or compare spec, plus `data.weights` to choose how often each source is used.